In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import plotly.express as px
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
df= pd.read_csv("Data/spotify-tracks-dataset.csv")
df.drop(columns= ['Unnamed: 0.1', 'Unnamed: 0'], inplace= True)

df['duration']= df.duration_ms /60_000

In [ ]:
df= pd.read_csv("Data/spotify-tracks-dataset.csv")
df.drop(columns= ['Unnamed: 0.1', 'Unnamed: 0'], inplace= True)

# Calculando duration en min
df['duration']= df.duration_ms /60_000
df.drop(columns= ['duration_ms'], inplace= True)

# Histogramas

<span style="color:red">NOTAS PARA EQUIPO:</span> 
- Puse dos versiones de histogramas, en teoría tienen la misma info
- Ojo con el histograma de instrumentalness del , ele eje Y está en escala logarítmica porque está muy cargado a la izquierda.

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go


def histogram_grid(df, cols_to_plot):


    fig = make_subplots(
        rows=3,
        cols=7,
        subplot_titles=["popularity"] + cols_to_plot,
        specs=[
            [{"colspan": 7}, None, None, None, None, None, None],
            [{}, {}, {}, {}, {}, {}, {}],
            [{}, {}, {}, {}, {}, {}, {}],
        ],
        vertical_spacing=0.08
    )

    # --------------------------------------------------
    # Popularity histogram (top row)
    # --------------------------------------------------
    fig.add_trace(
        go.Histogram(
            x=df["popularity"].dropna(),
            name="popularity",
            showlegend=False
        ),
        row=1,
        col=1
    )

    # --------------------------------------------------
    # Remaining variables
    # --------------------------------------------------
    for i, variable in enumerate(cols_to_plot):

        row = 2 + i // 7
        col = 1 + i % 7

        fig.add_trace(
            go.Histogram(
                x=df[variable].dropna(),
                name=variable,
                showlegend=False,
                
            ),
            row=row,
            col=col
            
        )

        # Logarithmic y-axis for instrumentalness
        if variable == "instrumentalness":
            fig.update_yaxes(
                type="log",
                row=row,
                col=col
            )

    fig.update_layout(
    height=1000,
    width=1400,
        template="simple_white",
        title="Variable distributions",
        bargap=0
    )

    return fig



cols_to_plot = [
    'explicit', 'key', 'mode', 'time_signature',
    'tempo', 'duration', 'loudness',
    'danceability', 'energy', 'speechiness',
    'acousticness', 'instrumentalness',
    'liveness', 'valence'
]

fig = histogram_grid(df, cols_to_plot)
fig.show()

In [ ]:
cols_to_plot= ['explicit', 'key', 'mode', 'time_signature','tempo', 'duration', 'loudness',
               'danceability', 'energy', 'speechiness', 'acousticness', 'instrumentalness', 
               'liveness','valence']

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# Create subplot titles
subplot_titles = []
stats = []

for col in cols_to_plot:
    if pd.api.types.is_numeric_dtype(df[col]):
        mean = df[col].mean()
        std = df[col].std()
        stats.append((mean, std))
        subplot_titles.append(col)
    else:
        stats.append((None, None))
        subplot_titles.append(col)

fig = make_subplots(
    rows=4,
    cols=4,
    subplot_titles=subplot_titles,
    vertical_spacing=0.12
)

# Add histograms
for i, col in enumerate(cols_to_plot):
    row = i // 4 + 1
    col_num = i % 4 + 1

    fig.add_trace(
        go.Histogram(
            x=df[col],
            name=col,
            showlegend=False
        ),
        row=row,
        col=col_num
    )

# Add mean/std annotations under titles
for i, (mean, std) in enumerate(stats):
    if mean is not None:
        fig.layout.annotations[i].text = (
            f"{cols_to_plot[i]}"
            f"<br><sup>μ={mean:.2f}, σ={std:.2f}</sup>"
        )

fig.update_layout(
    height=1200,
    width=1400,
    title_text="Histograms of Features",
    bargap=0.05,
)

fig.show()

# Correlación

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


# 2. Compute the pairwise correlation matrix
corr_matrix = df[cols_to_plot].corr()

# 3. Plot the heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True,  # Show correlation coefficients inside the squares
    cmap="coolwarm",  # Good diverging colormap (blue to red)
    vmin=-1,  # Force colorbar minimum to -1
    vmax=1,  # Force colorbar maximum to 1
    fmt=".2f",  # Format labels to 2 decimal places
    linewidths=0.5,  # Add tiny grid lines between blocks
)
plt.title("Correlation Matrix Plot")
plt.show()


# Popularidad por género
### Media, mediana, bandas de percentiles y porcentaje de canciones >=90 de popularidad

<span style="color:red">NOTAS PARA EQUIPO:</span> 

Hay muchas cosas pasando en esta gráfica:

- La linea azul es la mediana de la popularidad por género
- Linea naranja es popularidad promedio
- Las bandas azules son:
    - Banda obscura: La popularidad de los percentiles 25% - 75% (o sea que dentro de las bandas se encuentra el 50% de las canciones de cada género)
    - Banda clara: La popularidad de los percentiles 10% - 90% (o sea que dentro de las bandas se encuentra el 80% de las canciones de cada género)
  
      <br>

      
- Linea roja punteada: Porcentaje de canciones con popularidad mayor o igaul a 90. <span style="color:red"> Hay dos ejes Y</span> el de la derecha es para los porcentajes y el de la izquierda para todo lo demás. 

- Los géneros están ordenados de mayor a menor popularidad media.

<br><br>

- Se me hace muy interesante que después de detroit-techno (muy a la derecha), la mediana cae, pero el promedio sube mucho. Según yo eso significa que aunque muchas canciones son poco populares, hay canciones que está cargando muchísimo el promedio del género.


In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ----------------------------
# Compute statistics
# ----------------------------
genre_stats = (
    df.groupby("track_genre")["popularity"]
      .agg(
          n="size",
          median="median",
          mean="mean",
          q10=lambda x: x.quantile(0.10),
          q25=lambda x: x.quantile(0.25),
          q75=lambda x: x.quantile(0.75),
          q90=lambda x: x.quantile(0.90),
          pct_over_90=lambda x: (x > 90).mean() * 100
      )
      .sort_values("median", ascending=False)
      .reset_index()
)

# ----------------------------
# Figure with secondary y-axis
# ----------------------------
fig = make_subplots(
    specs=[[{"secondary_y": True}]]
)

# ----------------------------
# 10%-90% band
# ----------------------------
fig.add_trace(
    go.Scatter(
        x=genre_stats["track_genre"],
        y=genre_stats["q90"],
        mode="lines",
        line=dict(width=0),
        showlegend=False,
        hoverinfo="skip"
    ),
    secondary_y=False
)

fig.add_trace(
    go.Scatter(
        x=genre_stats["track_genre"],
        y=genre_stats["q10"],
        mode="lines",
        fill="tonexty",
        fillcolor="rgba(0,100,255,0.10)",
        line=dict(width=0),
        name="10%-90%",
        hoverinfo="skip"
    ),
    secondary_y=False
)

# ----------------------------
# 25%-75% band
# ----------------------------
fig.add_trace(
    go.Scatter(
        x=genre_stats["track_genre"],
        y=genre_stats["q75"],
        mode="lines",
        line=dict(width=0),
        showlegend=False,
        hoverinfo="skip"
    ),
    secondary_y=False
)

fig.add_trace(
    go.Scatter(
        x=genre_stats["track_genre"],
        y=genre_stats["q25"],
        mode="lines",
        fill="tonexty",
        fillcolor="rgba(0,100,255,0.25)",
        line=dict(width=0),
        name="25%-75%",
        hoverinfo="skip"
    ),
    secondary_y=False
)

# ----------------------------
# Mean
# ----------------------------
fig.add_trace(
    go.Scatter(
        x=genre_stats["track_genre"],
        y=genre_stats["mean"],
        mode="lines",
        name="Mean",
        line=dict(width=2),
        customdata=genre_stats[["n"]],
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Mean: %{y:.1f}<br>"
            "N: %{customdata[0]}<extra></extra>"
        )
    ),
    secondary_y=False
)

# ----------------------------
# Median (main signal)
# ----------------------------
fig.add_trace(
    go.Scatter(
        x=genre_stats["track_genre"],
        y=genre_stats["median"],
        mode="lines+markers",
        name="Median",
        line=dict(width=5),
        marker=dict(size=7),
        customdata=genre_stats[["n"]],
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Median: %{y:.1f}<br>"
            "N: %{customdata[0]}<extra></extra>"
        )
    ),
    secondary_y=False
)

# ----------------------------
# % Songs above 90 popularity
# ----------------------------
fig.add_trace(
    go.Scatter(
        x=genre_stats["track_genre"],
        y=genre_stats["pct_over_90"],
        mode="lines+markers",
        name="% > 90 Popularity",
        line=dict(width=3, dash="dot"),
        marker=dict(size=6),
        customdata=genre_stats[["n"]],
        hovertemplate=(
            "<b>%{x}</b><br>"
            "% > 90: %{y:.2f}%<br>"
            "N: %{customdata[0]}<extra></extra>"
        )
    ),
    secondary_y=True
)

# ----------------------------
# Layout
# ----------------------------
fig.update_layout(
    title="Popularity by Genre",
    xaxis_title="Genre (sorted by median popularity)",
    hovermode="x unified",
    xaxis_tickangle=-90,
    height=800,
    width=1600,
    template="plotly_white",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0
    )
)

fig.update_yaxes(
    title_text="Popularity",
    secondary_y=False
)

fig.update_yaxes(
    title_text="% Songs with Popularity > 90",
    secondary_y=True
)


fig.show()

# Histograma popularidad por género:


In [ ]:
import plotly.express as px


def genre_popularity_histogram(df, genre_list):

    df_plot = df[df["track_genre"].isin(genre_list)].copy()

    # Create legend labels with summary statistics
    label_map = {}

    for genre in genre_list:

        genre_data = df_plot.loc[
            df_plot["track_genre"] == genre,
            "popularity"
        ]

        mean_val = genre_data.mean()
        median_val = genre_data.median()

        label_map[genre] = (
            f"{genre} (med={median_val:.1f}, μ={mean_val:.1f})"
        )

    df_plot["genre_label"] = df_plot["track_genre"].map(label_map)

    fig = px.histogram(
        df_plot,
        x="popularity",
        color="genre_label",
        nbins=30,
        opacity=0.5,
        barmode="overlay",
        title=f"Popularity Distribution: {' vs '.join(genre_list)}"
    )

    fig.update_layout(
        xaxis_title="Popularity",
        yaxis_title="Count",
        template="plotly_white",
        legend_title="Genre",
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5
        )
    )

    return fig

 asdfasdfas `hola`

## <span style="color:red">NOTAS PARA EQUIPO:</span> 

- En estas gráficas puse géneros con popularidad mediana similar pero con promedio y bandas distintas.
- Yo puse los géneros que se me hizo interesante comparar, pero la idea es que juguemos con la gráfica de arriba (la que tiene un buen de lineas) y veamos que géneros nos llaman la atención comparar.
-  Si encuentran algún genero que les llamó la atención solo cambienle los valores de `genre_list` en cada celda. Yo puse 2 géneros por gráfica, pero pueden poner más si quieran :)


In [ ]:
genre_list= ['house', 'gospel' ]

genre_popularity_histogram(df, genre_list).show()

In [ ]:
genre_list= ['classical', 'reggae' ]

genre_popularity_histogram(df, genre_list).show()

In [ ]:
genre_list= [ 'latino', 'detroit-techno' ]

genre_popularity_histogram(df, genre_list).show()